# Làm sạch dữ liệu kinh tế vĩ mô theo quý của Việt Nam từ 1995 đến 2024

Notebook này thực hiện các bước:

1. Đọc dữ liệu từ file `vnm_data_2025_quarterly.csv`
2. Loại bỏ các dòng bị thiếu dữ liệu
3. Tính **Z-score thủ công** cho các biến số:
   - `GDP_Growth`
   - `Inflation`
   - `Interest_Rate`
4. Loại bỏ các quan sát ngoại lệ với điều kiện: |Z| < 3

5. Làm tròn dữ liệu đến 3 chữ số thập phân
6. Lưu dữ liệu sạch ra file `vnm_macro_final_cleaned.csv`

In [ ]:
import pandas as pd
import numpy as np

## 1. Đọc dữ liệu từ file CSV

Ở bước này, dữ liệu được đọc từ file `vnm_data_2025_quarterly.csv`.

- Cột đầu tiên được dùng làm **index**
- Index được chuyển sang định dạng **thời gian** để thuận tiện cho phân tích chuỗi thời gian

In [ ]:
try:
    df = pd.read_csv('vnm_data_2025_quarterly.csv', index_col=0)
    df.index = pd.to_datetime(df.index)
    print("Đọc file thành công.")
    print(df.head())
except FileNotFoundError:
    print("Không tìm thấy file 'vnm_data_2025_quarterly.csv'. Hãy kiểm tra lại tên file!")

## 2. Làm sạch dữ liệu

Quy trình làm sạch gồm 3 bước chính:

### Bước A. Loại bỏ các dòng bị thiếu dữ liệu

Do khi tính tốc độ tăng trưởng phần trăm, dòng đầu tiên thường bị thiếu giá trị nên cần loại bỏ.

### Bước B. Tính Z-score thủ công

Sử dụng công thức:

$$
Z = \frac{x - \bar{x}}{s}
$$

Trong đó:

- $x$: giá trị quan sát
- $\bar{x}$: giá trị trung bình
- $s$: độ lệch chuẩn

Chỉ giữ lại các quan sát thỏa mãn:

$$
-3 < Z < 3
$$

để loại bỏ các điểm ngoại lệ quá lớn.

### Bước C. Làm tròn dữ liệu

Sau khi làm sạch, dữ liệu được làm tròn đến 3 chữ số thập phân để trình bày đẹp hơn trong báo cáo.

In [ ]:
df_clean = df.dropna().copy()

In [ ]:
cols_to_clean = ['GDP_Growth', 'Inflation', 'Interest_Rate']

for col in cols_to_clean:
    mean = df_clean[col].mean()
    std = df_clean[col].std()
    
    z_score = (df_clean[col] - mean) / std
    
    df_clean = df_clean[np.abs(z_score) < 3]

In [ ]:
df_clean = df_clean.round(3)

## 3. Kiểm tra dữ liệu sau khi làm sạch

Sau khi loại bỏ các giá trị thiếu và ngoại lệ, ta kiểm tra:

- 5 dòng đầu tiên của bộ dữ liệu sạch
- Tổng số dòng còn lại

In [ ]:
print("--- Dữ liệu sau khi làm sạch thủ công ---")
print(df_clean.head())
print(f"\nSố lượng dòng còn lại: {len(df_clean)}")

## 4. Lưu dữ liệu sạch

Dữ liệu sau khi xử lý sẽ được lưu thành file:

`secondary_data_cleaned.csv`

File này có thể dùng để:

- nộp bài
- chạy mô hình
- tiếp tục phân tích thống kê hoặc dự báo

In [ ]:
df_clean.to_csv('secondary_data_cleaned.csv')
print("Đã lưu file: secondary_data_cleaned.csv")